# Finding faults in a solar plant
**GDGoC NUST x CYE AI Challenge 2026, team R.P.D.**

This notebook reads the three competition files, builds every feature, validates the pipeline by leaving out one day at a time, and
writes our two final submissions. It runs unchanged on Kaggle or locally, with fixed seeds, in about two minutes on a CPU.

| Section | |
|---|---|
| 1 to 4 | The problem, the physics, the features, how we validate |
| 5 | Configuration, data, features |
| 6 | Why a random split misleads |
| 7 to 10 | The model: two stages, synthetic faults, a voltage rule, a shading specialist |
| 11 | Using the shape of the day |
| 12 to 14 | Results, submission files, summary |

## 1. The problem
Each row is one reading from a small solar plant: voltage, current, irradiance (G) and panel temperature (T). For each reading we name the
plant's state: normal, shading, short circuit, degradation or open circuit. The metric is macro F1, so a class with 360 rows
counts as much as one with 5,000.

The rows are shuffled and the faults are rare (0.5 to 1.6 % of rows), present on only a few days. What we are really asked
is whether a model can recognise a fault on a day it has never seen.

## 2. The physics
The plant has two strings of eight modules. The recorded voltage is the mean of the two strings, the current is their sum. In a healthy
plant, current follows irradiance, and voltage falls slowly with temperature:

$$I \approx aG + bG(T-25), \qquad V \approx c_0 + c_1T + c_2\log G$$

Each fault leaves its own mark against that expectation.

| Fault | What happens | What we see |
|---|---|---|
| Open circuit | one string drops out | V and I fall to about half |
| Short circuit | 2 of 8 modules bypassed in one string | V falls to (6/8 + 8/8)/2 = 0.875, I unchanged |
| Degradation | added series resistance | V falls in proportion to I |
| Shading | part of the array in shadow | I falls below what G implies |

## 3. Features
We fit expected current and voltage on normal rows only, then describe every reading by its distance from them: the ratios `I_ratio`,
`V_ratio`, `P_ratio`, the deficit `dV_frac`, and a hidden resistance `R_like = (V expected - V) / I`. A second set of expectations
comes from small LightGBM regressors on (G, T), which capture the curvature that the formulas miss. `G_cap = min(G, 800)` closes one
trap: all training short circuits happened in bright sun, and a model left alone learns "sunny means short".

## 4. Validation
We hide one day, train on the other twelve, predict the hidden day, and repeat thirteen times. The pooled predictions give one macro F1,
and every decision was made on it. The fitted coefficients and classifiers are rebuilt inside each fold. The expected-value regressors are fitted
once per day on the other twelve days, so no row is scored by a regressor that saw its own day.

## 5. Configuration, data, features
`FINAL_STAGES` names the two stages written as submissions. Switches exist so that any step can be turned off and its effect measured.

In [1]:
import os, sys, glob, random, time, warnings
for _v in ('OMP_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'MKL_NUM_THREADS'):   # parallelism = folds, not BLAS threads
    os.environ.setdefault(_v, '1')
from pathlib import Path
import numpy as np, pandas as pd
import lightgbm as lgb, sklearn, joblib
from joblib import Parallel, delayed
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score, confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
warnings.filterwarnings('ignore')
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)


def flag(env, default):
    return bool(int(os.environ.get(env, int(default))))


# switches
RUN_SPLIT_DEMO = flag('NB_SPLIT_DEMO', True)     # section 6
USE_T2_AUGMENT = flag('NB_T2', True)             # section 8: synthetic faults
T2_ALPHA = 0.40                                  # weight inside stage B
T2_SEEDS = [0, 1, 2]
USE_V_GUARD = flag('NB_VGUARD', True)            # section 9: voltage rule
V_GUARD = 180.0
USE_T8_SPECIALIST = flag('NB_T8', True)          # section 10: shading specialist
T8_W = 0.30                                      # weight of the specialist

# stage names grow with the pipeline: 'etB3', '+ _T2a40' (synthetic faults), '+ _T8w30' (specialist), then section 11
FINAL_STAGES = os.environ.get('NB_FINAL', 'three_opinions_gauss,three_opinions_smooth').split(',')

SEED = 42
SEEDS = [42, 43, 44]
random.seed(SEED); np.random.seed(SEED)
# every model is single-threaded inside its fold, so results do not depend on the number of cores
N_JOBS = int(os.environ.get('NB_N_JOBS', -1))
T0 = time.time()


def find_data_dir():
    cands = ['/kaggle/input/gdgoc-cye-ai-2026/', '../data/', 'data/']
    cands += sorted(set(os.path.dirname(p) + '/' for p in glob.glob('/kaggle/input/**/train.csv', recursive=True)))
    for c in cands:
        if os.path.exists(os.path.join(c, 'train.csv')) and os.path.exists(os.path.join(c, 'test.csv')):
            return c
    raise FileNotFoundError('train.csv / test.csv not found')


DATA = find_data_dir()
ON_KAGGLE = DATA.startswith('/kaggle/')
print('data dir:', DATA, '| kaggle:', ON_KAGGLE, '| n_jobs:', N_JOBS, '| cores:', os.cpu_count())
print('switches: T2 augment', USE_T2_AUGMENT, '| voltage guard', USE_V_GUARD, '| T8 specialist', USE_T8_SPECIALIST,
      '| final stages', FINAL_STAGES)
print('python', sys.version.split()[0], '| lightgbm', lgb.__version__, '| scikit-learn', sklearn.__version__,
      '| numpy', np.__version__, '| pandas', pd.__version__, '| joblib', joblib.__version__)

LABELS = ['normal', 'shading', 'short_circuit', 'degradation', 'open_circuit']
L2I = {l: i for i, l in enumerate(LABELS)}
RAW = ['dc_voltage', 'dc_current', 'irradiance', 'panel_temp']
BETA = -0.0045  # voltage temperature coefficient (1/K), used for V_tc

data dir: ../data/ | kaggle: False | n_jobs: -1 | cores: 20
switches: T2 augment True | voltage guard True | T8 specialist True | final stages ['three_opinions_gauss', 'three_opinions_smooth']
python 3.12.14 | lightgbm 4.7.0 | scikit-learn 1.9.1 | numpy 2.5.5 | pandas 3.1.1 | joblib 1.6.0


### 5.1 Physics features

In [2]:
def phys_fit(df):
    """Parametric expected-current / expected-voltage models fitted on normal rows (G > 150)."""
    n = df[(df.label == 'normal') & (df.irradiance > 150)]
    G, T = n.irradiance.values, n.panel_temp.values
    ci = np.linalg.lstsq(np.c_[G, G * (T - 25)], n.dc_current.values, rcond=None)[0]
    cv = np.linalg.lstsq(np.c_[np.ones_like(T), T, np.log(G)], n.dc_voltage.values, rcond=None)[0]
    return ci, cv


def param_feats(coefs, df):
    ci, cv = coefs
    G, T, V, I = df.irradiance.values, df.panel_temp.values, df.dc_voltage.values, df.dc_current.values
    o = pd.DataFrame(index=df.index)
    o['P'] = V * I
    o['I_over_G'] = I / G
    o['P_over_G'] = V * I / G
    o['V_tc'] = V / (1 + BETA * (T - 25))
    Ie = ci[0] * G + ci[1] * G * (T - 25)
    Ve = cv[0] + cv[1] * T + cv[2] * np.log(G)
    o['I_ratio'] = I / Ie
    o['V_ratio'] = V / Ve
    o['P_ratio'] = o.I_ratio * o.V_ratio
    o['dV_frac'] = 1 - o.V_ratio
    o['R_like'] = (Ve - V) / np.maximum(I, 0.5)
    o['lowG'] = (G < 150).astype(int)
    o['notrack'] = (o.I_ratio < 0.2).astype(int)
    return o


def gbm_expect(train_df, apply_df, nthreads=1):
    """Non-parametric expected I and V given (G, T), LightGBM regressors trained on normal rows."""
    n = train_df[train_df.label == 'normal']
    prm = dict(n_estimators=200, learning_rate=0.05, num_leaves=15, min_child_samples=50, verbose=-1,
               random_state=SEED, n_jobs=nthreads)
    X = apply_df[['irradiance', 'panel_temp']]
    mi = lgb.LGBMRegressor(**prm).fit(n[['irradiance', 'panel_temp']], n.dc_current)
    mv = lgb.LGBMRegressor(**prm).fit(n[['irradiance', 'panel_temp']], n.dc_voltage)
    return mi.predict(X), mv.predict(X)


def gbm_feats(df, Ie, Ve):
    o = pd.DataFrame(index=df.index)
    o['I_ratio_g'] = df.dc_current.values / np.maximum(Ie, 0.2)
    o['V_ratio_g'] = df.dc_voltage.values / Ve
    o['I_res_g'] = df.dc_current.values - Ie
    o['V_res_g'] = df.dc_voltage.values - Ve
    o['R_like_g'] = (Ve - df.dc_voltage.values) / np.maximum(df.dc_current.values, 0.5)
    return o


def add_extra(F):
    """Extra ratio features from the learned expectations.
    Stage B uses dVg, logV_g, R_exp_g and G_cap; the specialist uses the rest."""
    F = F.copy()
    Ie = F.dc_current - F.I_res_g
    Ve = F.dc_voltage - F.V_res_g
    F['dVg'] = 1 - F.V_ratio_g                                   # voltage deficit vs expectation
    F['dV_over_I'] = F.dVg / np.maximum(F.I_ratio_g, 0.05)
    F['R_exp_g'] = (Ve - F.dc_voltage) / np.maximum(Ie, 0.5)     # series-resistance proxy at the expected current
    F['logI_g'] = np.log(np.maximum(F.I_ratio_g, 1e-3))
    F['logV_g'] = np.log(np.maximum(F.V_ratio_g, 1e-3))
    F['I_minus_V'] = F.I_ratio_g - F.V_ratio_g
    F['P_ratio_g'] = F.I_ratio_g * F.V_ratio_g
    F['dV_over_I_par'] = F.dV_frac / np.maximum(F.I_ratio, 0.05)
    Ie_p = F.dc_current / np.maximum(F.I_ratio, 1e-3)
    F['R_exp'] = F.dV_frac * (F.dc_voltage / np.maximum(F.V_ratio, 1e-3)) / np.maximum(Ie_p, 0.5)
    F['V_gap_pg'] = F.V_ratio - F.V_ratio_g
    F['logR'] = np.sign(F.R_like_g) * np.log1p(np.abs(F.R_like_g))
    F['G_cap'] = F.irradiance.clip(0, 800)                       # cap G: no "short only at G > 900" shortcut
    return F


PARAM = ['P', 'I_over_G', 'P_over_G', 'V_tc', 'I_ratio', 'V_ratio', 'P_ratio', 'dV_frac', 'R_like', 'lowG', 'notrack']
GBMF = ['I_ratio_g', 'V_ratio_g', 'I_res_g', 'V_res_g', 'R_like_g']
PHYS = RAW + PARAM + GBMF
VB = ['V_ratio_g', 'V_ratio', 'dVg', 'logV_g', 'R_like_g', 'R_exp_g', 'R_like', 'G_cap', 'lowG']  # stage-B features

### 5.2 Loading and cleaning
Two training rows (3986 and 8152) are labelled degradation but show half the expected voltage, which only an open circuit can do. We
drop them and keep everything else. Test rows are never edited.

In [3]:
tr = pd.read_csv(DATA + 'train.csv'); te = pd.read_csv(DATA + 'test.csv')
ss = pd.read_csv(DATA + 'sample_submission.csv')
ci0, cv0 = phys_fit(tr)
vr0 = tr.dc_voltage / (cv0[0] + cv0[1] * tr.panel_temp + cv0[2] * np.log(tr.irradiance))
bad = (tr.label == 'degradation') & (vr0 < 0.7)
print('dropped ids:', tr.id[bad].tolist())
tr = tr[~bad].reset_index(drop=True)
y = tr.label.map(L2I).values
days = tr.day.values
DAYS = np.unique(days)
folds = [(np.where(days != d)[0], np.where(days == d)[0]) for d in DAYS]
print('train', tr.shape, '| test', te.shape)
print(tr.groupby(['day', 'label']).size().unstack(fill_value=0))

dropped ids: [3986, 8152]
train (44015, 7) | test (9617, 5)
label  degradation  normal  open_circuit  shading  short_circuit
day                                                             
0                0    1435             0     1491              0
1                0    1453             0     1479              0
2                0    1848             0     1254              0
3                0    1451             0     1537              0
4                0    1955             0     1237              0
5                0    2064             0     1091              0
6                0    2430             0     1360              0
7                0    2501             0     1155              0
8              178    2672             1     1042              0
9              270    2301             0     1374              0
10               0    2533             0     1173              0
11             122    2040           120      991            123
12             119    1904    

### 5.3 Learned expectations
Expected current and voltage for a training day come from models fitted on the other twelve days; the test rows use all thirteen.

In [4]:
Ie_tr = np.zeros(len(tr)); Ve_tr = np.zeros(len(tr))
res = Parallel(n_jobs=N_JOBS, backend='loky')(
    delayed(lambda d: (d, gbm_expect(tr[days != d], tr[days == d])))(d) for d in DAYS)
for d, (ie, ve) in res:
    Ie_tr[days == d], Ve_tr[days == d] = ie, ve
Ie_te, Ve_te = gbm_expect(tr, te)
G_tr, G_te = gbm_feats(tr, Ie_tr, Ve_tr), gbm_feats(te, Ie_te, Ve_te)


def fold_features(fit_idx, rows='train', idx=None):
    """Full feature frame; the parametric coefficients are fitted on normal rows of tr.iloc[fit_idx] only."""
    coefs = phys_fit(tr.iloc[fit_idx])
    if rows == 'train':
        df, G = tr.iloc[idx], G_tr.iloc[idx]
    else:
        df, G = te, G_te
    return add_extra(pd.concat([df[RAW], param_feats(coefs, df), G], axis=1))


print(pd.concat([fold_features(np.arange(len(tr)), 'train', np.arange(len(tr)))[PHYS + VB[2:4] + ['R_exp_g']],
                 tr.label], axis=1).groupby('label').median().T.round(3))

label       degradation    normal  open_circuit   shading  short_circuit
dc_voltage      245.671   271.118       130.319   284.246        230.404
dc_current       17.434    14.394         9.085     7.150         17.657
irradiance      962.727   819.494       958.566   500.047        944.013
panel_temp       51.238    41.108        49.995    32.552         46.836
P              4210.483  3892.691      1177.306  2005.759       4058.245
I_over_G          0.018     0.018         0.010     0.014          0.019
P_over_G          4.499     4.747         1.240     4.017          4.285
V_tc            278.567   292.852       145.876   295.594        255.321
I_ratio           0.996     0.998         0.520     0.813          1.033
V_ratio           0.947     0.999         0.496     1.010          0.870
P_ratio           0.948     0.996         0.261     0.839          0.901
dV_frac           0.053     0.001         0.504    -0.010          0.130
R_like            0.804     0.026        14.428    

### 5.4 Scoring helpers

In [5]:
f32 = lambda a: np.asarray(a, np.float32)   # probabilities are stored as float32 between blend levels
K = len(folds)
STAGES = {}                                 # name -> dict(oof=(n_train, 5), test=(n_test, 5)); filled by sections 7-11


def class_f1(P, rows=slice(None)):
    return f1_score(y[rows], P[rows].argmax(1), average=None, labels=range(5))


def report(name, P):
    pc = class_f1(P)
    print(f'{name:34s} macroF1={pc.mean():.4f} | ' + ' '.join(f'{l[:5]}={v:.3f}' for l, v in zip(LABELS, pc)))
    return pc.mean()


def run_parallel(tasks):
    """Run the fold fits; returns {(model, seed): (oof, test)}, test = mean of the 13 fold models (fold order)."""
    t1, runs = time.time(), {}
    for v, s, k, pb, pt in Parallel(n_jobs=N_JOBS, backend='loky')(tasks):
        oof, test = runs.setdefault((v, s), (np.zeros((len(tr),) + pb.shape[1:]), np.zeros((len(te),) + pt.shape[1:])))
        oof[folds[k][1]] = pb; test += pt / K
    print(f'{len(tasks)} fold fits in {time.time() - t1:.0f}s')
    return runs


def voltage_guard(P, V):
    """Section 9: rows with dc_voltage < V_GUARD become open_circuit (one-hot), all other rows are untouched."""
    out = P.copy()
    m = np.asarray(V) < V_GUARD
    out[m] = 0; out[m, L2I['open_circuit']] = 1
    return out


def add_stage(name, oof, test):
    """Register the output of one pipeline step (after the voltage guard) and print its LODO score."""
    g_oof = voltage_guard(oof, tr.dc_voltage) if USE_V_GUARD else oof
    g_test = voltage_guard(test, te.dc_voltage) if USE_V_GUARD else test
    STAGES[name] = dict(oof=g_oof, test=g_test,
                        guard_oof=int((g_oof.argmax(1) != oof.argmax(1)).sum()),
                        guard_test=int((g_test.argmax(1) != test.argmax(1)).sum()))
    report('STAGE ' + name, g_oof)
    if USE_V_GUARD:
        print(f'   labels changed by the voltage guard: OOF {STAGES[name]["guard_oof"]}, test {STAGES[name]["guard_test"]}')

## 6. Why a random split misleads
The same baseline under two protocols. Readings ten seconds apart are near copies, so a random split lets the model remember rather than
generalise. The test set is made of unseen days, so only the second number means anything.

In [6]:
def run_fold_baseline(tag, a, b):
    Xa, Xb = fold_features(a, 'train', a)[PHYS], fold_features(a, 'train', b)[PHYS]
    m = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20, subsample=0.8,
                           subsample_freq=1, colsample_bytree=0.8, class_weight='balanced', verbose=-1,
                           random_state=SEED, n_jobs=1).fit(Xa, y[a])
    return tag, b, m.classes_[m.predict_proba(Xb).argmax(1)]


if RUN_SPLIT_DEMO:
    t1 = time.time()
    rnd = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))
    demo = {'random 5-fold': np.zeros(len(y), int), 'leave-one-day-out': np.zeros(len(y), int)}
    tasks = [delayed(run_fold_baseline)('random 5-fold', a, b) for a, b in rnd]
    tasks += [delayed(run_fold_baseline)('leave-one-day-out', a, b) for a, b in folds]
    for tag, b, p in Parallel(n_jobs=N_JOBS, backend='loky')(tasks):
        demo[tag][b] = p
    rows = []
    for tag, p in demo.items():
        pc = f1_score(y, p, average=None, labels=range(5))
        rows.append(dict(split=tag, macro_f1=pc.mean(), **dict(zip(LABELS, pc))))
    print(pd.DataFrame(rows).set_index('split').round(4))
    print(f'({time.time() - t1:.0f}s)')

                   macro_f1  normal  shading  short_circuit  degradation  open_circuit
split                                                                                 
random 5-fold        0.9921  0.9925   0.9879         0.9898       0.9906        1.0000
leave-one-day-out    0.8753  0.9356   0.9144         0.9367       0.5938        0.9959
(6s)


## 7. Base model: two questions instead of one
Shading is about current, electrical faults are about voltage, so we ask two separate questions.

* **Stage A**, LightGBM: is this reading shaded?
* **Stage B**, ExtraTrees on voltage features only: if not shaded, is it normal, short, degraded or open?

The answers combine as `P(shading) = A` and `(1 - A) * B` for the rest, averaged over three variants of stage A and three seeds.

In [7]:
PA_BASE = dict(n_estimators=200, learning_rate=0.05, num_leaves=31, min_child_samples=20, subsample=0.8,
               subsample_freq=1, colsample_bytree=0.8, verbose=-1)
VARIANTS = {   # name -> (stage-A features, stage-A LightGBM params)
    'Aphys_n200': (PHYS, {}),
    'Araw_param': (RAW + PARAM, dict(n_estimators=400)),
    'Aphys_nl15': (PHYS, dict(n_estimators=400, num_leaves=15, min_child_samples=50)),
}
B_TO_5 = [0, 2, 3, 4]                       # stage-B classes -> label index
Y5_TO_B = np.array([0, -1, 1, 2, 3])        # label index -> stage-B class (shading excluded)


def fit_predict_shade_split(Xa, ya, Xlist, featsA, paramsA, seed):
    """Stage A: LightGBM shading-vs-rest. Stage B: ExtraTrees on non-shading rows with voltage-only features VB."""
    mA = lgb.LGBMClassifier(**{**PA_BASE, **paramsA, 'random_state': seed, 'n_jobs': 1}).fit(
        Xa[featsA], (ya == 1).astype(int))
    f = ya != 1
    mB = ExtraTreesClassifier(n_estimators=300, min_samples_leaf=20, max_features=0.7, class_weight=None,
                              random_state=seed, n_jobs=1).fit(np.asarray(Xa.loc[f, VB], float), Y5_TO_B[ya[f]])
    out = []
    for X in Xlist:
        ps = mA.predict_proba(X[featsA])[:, 1]
        PB = np.zeros((len(X), 4)); PB[:, mB.classes_] = mB.predict_proba(np.asarray(X[VB], float))
        p = np.zeros((len(X), 5)); p[:, 1] = ps; p[:, B_TO_5] = (1 - ps)[:, None] * PB
        out.append(p)
    return out


def run_fold(vname, seed, k):
    a, b = folds[k]
    featsA, paramsA = VARIANTS[vname]
    Xa, Xb, Xt = fold_features(a, 'train', a), fold_features(a, 'train', b), fold_features(a, 'test')
    pb, pt = fit_predict_shade_split(Xa, y[a], [Xb, Xt], featsA, paramsA, seed)
    return vname, seed, k, pb, pt


RUNS = run_parallel([delayed(run_fold)(v, s, k) for v in VARIANTS for s in SEEDS for k in range(K)])
VAR_OOF, VAR_TEST = {}, {}
for v in VARIANTS:
    VAR_OOF[v] = f32(np.mean([f32(RUNS[(v, s)][0]) for s in SEEDS], 0))
    VAR_TEST[v] = f32(np.mean([f32(RUNS[(v, s)][1]) for s in SEEDS], 0))
    report(f'{v} (3 seeds)', VAR_OOF[v])
CUR = 'etB3'                                 # name of the latest stage; every section below starts from STAGES[CUR]
add_stage(CUR, f32(np.mean([VAR_OOF[v] for v in VARIANTS], 0)), f32(np.mean([VAR_TEST[v] for v in VARIANTS], 0)))

117 fold fits in 20s
Aphys_n200 (3 seeds)               macroF1=0.9474 | norma=0.948 shadi=0.921 short=0.980 degra=0.888 open_=1.000
Araw_param (3 seeds)               macroF1=0.9481 | norma=0.949 shadi=0.922 short=0.982 degra=0.888 open_=1.000
Aphys_nl15 (3 seeds)               macroF1=0.9484 | norma=0.951 shadi=0.925 short=0.982 degra=0.887 open_=0.998
STAGE etB3                         macroF1=0.9493 | norma=0.951 shadi=0.926 short=0.982 degra=0.888 open_=1.000
   labels changed by the voltage guard: OOF 0, test 0


## 8. Synthetic faults
The training faults come from two to four days, too few to show how they look on another. Since the wiring tells us exactly what each fault
does, we make more: inside each fold we take normal rows of the training days and apply the fault's effect, calibrated on that fold's real
faults. A LightGBM trained on real and synthetic rows is mixed into stage B with weight 0.4.

This was our first upload (public 0.942).

In [8]:
AUG_CFG = dict(n_per_class=6000, w=0.2, norm=True, gmin=800, gmax=1100, shrink=0.5, r_lo=0.5, r_hi=1.6, vm_ext=0.4,
               deg_vmax=0.952, i_jit=(0.96, 1.08), classes=('open_circuit', 'short_circuit', 'degradation'))
T2_LGB = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, min_child_samples=20, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, class_weight=None, verbose=-1)


def calibrate(a):
    """Per-fault-class ratio distributions from the fold's training rows only."""
    t, G = tr.iloc[a], G_tr.iloc[a]
    cal = {}
    for lab in ['open_circuit', 'short_circuit', 'degradation']:
        m = (t.label == lab).values
        if m.sum() < 5:
            continue
        vr, ir, rl = G.V_ratio_g.values[m], G.I_ratio_g.values[m], G.R_like_g.values[m]
        lo, hi = np.quantile(vr, [0.05, 0.95]); k = (vr >= lo) & (vr <= hi)   # drop transition rows
        cal[lab] = dict(vr=vr[k], ir=ir[k], rl=rl[k], vr_med=np.median(vr[k]), ir_med=np.median(ir[k]),
                        rl_q=np.quantile(rl[k], [0.1, 0.5, 0.9]), vr_q=np.quantile(vr[k], [0.1, 0.5, 0.9]))
    return cal


def _draw(rng, arr, med, shrink, n):
    return med + shrink * (rng.choice(arr, n) - med)


def synth(a, cfg, rng):
    t, G = tr.iloc[a], G_tr.iloc[a]
    cal = calibrate(a)
    lab = t.label.values
    base = (lab == 'normal') & (t.irradiance.values > cfg['gmin']) & (t.irradiance.values < cfg['gmax'])
    gbin = np.digitize(t.irradiance.values, np.arange(cfg['gmin'], cfg['gmax'] + 1, 100))
    rows, ys = [], []
    for c in cfg['classes']:
        if c not in cal:
            continue
        n = cfg['n_per_class']
        bins = list(np.unique(gbin[base]))                      # stratified over 100 W/m2 irradiance bins
        per = int(np.ceil(n / len(bins)))
        idx = np.concatenate([rng.choice(np.where(base & (gbin == b))[0], per) for b in bins])
        idx = rng.permutation(idx)[:n]
        V, I = t.dc_voltage.values[idx], t.dc_current.values[idx]
        Ve, Ie = V - G.V_res_g.values[idx], I - G.I_res_g.values[idx]
        k = cal[c]; sh = cfg['shrink']
        im = _draw(rng, k['ir'], k['ir_med'], sh, len(idx))     # current multiplier
        if c in ('degradation', 'short_circuit'):
            im = im * rng.uniform(cfg['i_jit'][0], cfg['i_jit'][1], len(idx))
        if c in ('open_circuit', 'short_circuit'):
            vm = _draw(rng, k['vr'], k['vr_med'], sh, len(idx))  # voltage multiplier
            Vn, In = V * vm, I * im
        else:  # degradation: multiplicative voltage deficit, V_ratio capped at deg_vmax
            In = I * im
            q, vq = k['rl_q'], k['vr_q']
            _ = rng.uniform(cfg['r_lo'] * q[0], cfg['r_hi'] * q[2], len(idx))  # keeps the random stream fixed
            vm = rng.uniform(1 - (1 + cfg['vm_ext']) * (1 - vq[0]), 1 - (1 - cfg['vm_ext']) * (1 - vq[2]), len(idx))
            Vn = V * vm
            keep = Vn / Ve <= cfg['deg_vmax']
            Vn, In, idx, Ve, Ie = Vn[keep], In[keep], idx[keep], Ve[keep], Ie[keep]
        df = pd.DataFrame({'dc_voltage': Vn, 'dc_current': In, 'irradiance': t.irradiance.values[idx],
                           'panel_temp': t.panel_temp.values[idx], 'label': c})
        df['_Ie'], df['_Ve'] = Ie, Ve
        rows.append(df); ys.append(np.full(len(df), L2I[c]))
    return pd.concat(rows, ignore_index=True), np.concatenate(ys)


def run_fold_t2(s, k):
    """T2 LightGBM on real + synthetic rows of fold k (synthetic seed s, LightGBM seed 42 + s)."""
    a, b = folds[k]
    cfg = AUG_CFG
    Xa, Xb, Xt = (fold_features(a, 'train', a)[PHYS], fold_features(a, 'train', b)[PHYS], fold_features(a, 'test')[PHYS])
    ya = y[a]
    bw = len(ya) / (5 * np.maximum(np.bincount(ya, minlength=5).astype(float), 1))
    wa = bw[ya]
    S, ys = synth(a, cfg, np.random.default_rng(s * 1000 + int(DAYS[k])))
    coefs = phys_fit(tr.iloc[a])
    Xs = pd.concat([S[RAW], param_feats(coefs, S), gbm_feats(S, S._Ie.values, S._Ve.values)], axis=1)[PHYS]
    ws = cfg['w'] * bw[ys]
    for c in np.unique(ys):   # keep each class's total weight = its balanced mass, split real / synthetic
        mr, ms = wa[ya == c].sum(), ws[ys == c].sum()
        tot = len(ya) / 5; share_real = mr / (mr + ms)
        wa[ya == c] = wa[ya == c] * (tot * share_real / mr)
        ws[ys == c] = ws[ys == c] * (tot * (1 - share_real) / ms)
    X = pd.concat([Xa, Xs], ignore_index=True)
    m = lgb.LGBMClassifier(**{**T2_LGB, 'random_state': SEED + s, 'n_jobs': 1}).fit(
        X, np.concatenate([ya, ys]), sample_weight=np.concatenate([wa, ws]))
    out = []
    for Z in (Xb, Xt):
        p = np.zeros((len(Z), 5)); p[:, m.classes_] = m.predict_proba(Z); out.append(p)
    return ('t2', s, k, out[0], out[1])


NS = [0, 2, 3, 4]   # non-shading classes


def into_B(p, q, alpha):
    """Keep P(shading) of p; mix q's conditional non-shading distribution into p's stage-B distribution."""
    ps = p[:, 1]
    PB = p[:, NS] / np.maximum(1 - ps, 1e-9)[:, None]
    QB = q[:, NS] / np.maximum(q[:, NS].sum(1), 1e-9)[:, None]
    out = np.zeros_like(p); out[:, 1] = ps; out[:, NS] = (1 - ps)[:, None] * ((1 - alpha) * PB + alpha * QB)
    return out


if USE_T2_AUGMENT:
    RUNS_T2 = run_parallel([delayed(run_fold_t2)(s, k) for s in T2_SEEDS for k in range(K)])
    T2_OOF = f32(np.mean([RUNS_T2[('t2', s)][0] for s in T2_SEEDS], 0))
    T2_TEST = f32(np.mean([RUNS_T2[('t2', s)][1] for s in T2_SEEDS], 0))
    report('T2 augmented LightGBM alone', T2_OOF)
    prev, CUR = CUR, f'{CUR}_T2a{int(round(T2_ALPHA * 100))}'
    add_stage(CUR, into_B(STAGES[prev]['oof'], T2_OOF, T2_ALPHA), into_B(STAGES[prev]['test'], T2_TEST, T2_ALPHA))

39 fold fits in 21s
T2 augmented LightGBM alone        macroF1=0.9372 | norma=0.941 shadi=0.915 short=0.982 degra=0.848 open_=1.000
STAGE etB3_T2a40                   macroF1=0.9536 | norma=0.952 shadi=0.926 short=0.984 degra=0.907 open_=1.000
   labels changed by the voltage guard: OOF 0, test 0


## 9. A voltage rule
Only an open circuit can halve the voltage; the worst short circuit still keeps 7/8 of it. Below 180 V we therefore call open
circuit whatever the models say. In every fold, all open-circuit training rows lie under the line and everything else above it. The rule changes
no label today. It guards against one borderline row, whose flip would cost a quarter of day 8's macro F1.

In [9]:
V_tr, is_open = tr.dc_voltage.values, y == L2I['open_circuit']
print(f'train: max V of open_circuit rows = {V_tr[is_open].max():.1f} V | min V of all other rows = {V_tr[~is_open].min():.1f} V')
gap = [(V_tr[a][is_open[a]].max(), V_tr[a][~is_open[a]].min()) for a, _ in folds]      # per fold: training days only
print('gap learned from the training days of each fold:', f'max(open) {max(g[0] for g in gap):.1f} < {V_GUARD:.0f} <',
      f'min(other) {min(g[1] for g in gap):.1f} ->', 'holds in all folds' if all(lo < V_GUARD < hi for lo, hi in gap) else 'FAILS')
for part, V in (('train', V_tr), ('test', te.dc_voltage.values)):
    print(f'{part}: rows below {V_GUARD:.0f} V: {(V < V_GUARD).sum()} | rows within +-25 V of the threshold: {(np.abs(V - V_GUARD) < 25).sum()}')
for n, s in STAGES.items():
    print(f'labels changed by the guard in stage {n}: OOF {s["guard_oof"]}, test {s["guard_test"]}')

train: max V of open_circuit rows = 138.7 V | min V of all other rows = 204.8 V
gap learned from the training days of each fold: max(open) 138.7 < 180 < min(other) 204.8 -> holds in all folds
train: rows below 180 V: 242 | rows within +-25 V of the threshold: 1
test: rows below 180 V: 360 | rows within +-25 V of the threshold: 0
labels changed by the guard in stage etB3: OOF 0, test 0
labels changed by the guard in stage etB3_T2a40: OOF 0, test 0


## 10. A shading specialist
What remains is mostly normal against shading, on days that differ in temperature and soiling. A second model sees only those two classes
and measures each reading against its own day's median. It takes a 0.3 share of that decision.

The test days are mixed together, so there we use the median of the whole pile. That is weaker than in validation, and the gain
here is a little optimistic: about +0.001 when days are pooled.

In [10]:
NEW_T1 = ['dVg', 'dV_over_I', 'R_exp_g', 'logI_g', 'logV_g', 'I_minus_V', 'P_ratio_g', 'dV_over_I_par', 'R_exp',
          'V_gap_pg', 'logR', 'G_cap']
TP = ['Tres', 'Tres_over_G', 'absLogIr', 'absLogIrp', 'IrHi', 'IrLo', 'V_over_logG', 'I_over_sqrtG']
T8_FEATS = list(dict.fromkeys(PHYS + NEW_T1 + TP + ['T_dayrel', 'Ir_dayrel', 'Vr_dayrel', 'Tres_day']))
T8_LGB = dict(n_estimators=500, learning_rate=0.04, num_leaves=15, min_child_samples=200, subsample=0.8,
              subsample_freq=1, colsample_bytree=0.8, reg_lambda=10.0, verbose=-1)


def day_stats(df, Gs):
    hi = df.irradiance.values > 800; h7 = df.irradiance.values > 700
    return np.array([np.median(df.panel_temp.values[hi]), np.median(Gs.I_ratio_g.values[h7]),
                     np.median(Gs.V_ratio_g.values[h7])])


def t8_features(a, rows, idx, coefT):
    F = fold_features(a, rows, idx)
    if rows == 'test':
        S = np.tile(day_stats(te, G_te), (len(F), 1))
    else:
        dd = days[idx]; S = np.zeros((len(idx), 3))
        for d in np.unique(dd):
            mm = days == d
            S[dd == d] = day_stats(tr[mm], G_tr[mm])
    G, T = F.irradiance.values, F.panel_temp.values
    Te = coefT[0] + coefT[1] * G + coefT[2] * np.sqrt(np.maximum(G, 0))
    F['Tres'] = T - Te
    F['Tres_over_G'] = F.Tres / np.maximum(G, 20)
    F['absLogIr'] = np.abs(np.log(np.maximum(F.I_ratio_g, 1e-3)))
    F['absLogIrp'] = np.abs(np.log(np.maximum(F.I_ratio, 1e-3)))
    F['IrHi'] = np.log(np.maximum(F.I_ratio_g, 1e-3)).clip(0, None)
    F['IrLo'] = (-np.log(np.maximum(F.I_ratio_g, 1e-3))).clip(0, None)
    F['V_over_logG'] = F.dc_voltage / np.log(np.maximum(G, 2))
    F['I_over_sqrtG'] = F.dc_current / np.sqrt(np.maximum(G, 1))
    F['T_dayrel'] = F.panel_temp.values - S[:, 0]
    F['Ir_dayrel'] = F.I_ratio_g.values / S[:, 1]
    F['Vr_dayrel'] = F.V_ratio_g.values / S[:, 2]
    Te850 = coefT[0] + coefT[1] * 850 + coefT[2] * np.sqrt(850)
    F['Tres_day'] = F.T_dayrel - (Te - Te850)
    return F[T8_FEATS]


def run_fold_t8(k):
    a, b = folds[k]
    n = tr.iloc[a]; n = n[n.label == 'normal']
    G = n.irradiance.values
    coefT = np.linalg.lstsq(np.c_[np.ones_like(G), G, np.sqrt(G)], n.panel_temp.values, rcond=None)[0]  # expected T(G)
    Fa, Fb, Ft = t8_features(a, 'train', a, coefT), t8_features(a, 'train', b, coefT), t8_features(a, 'test', None, coefT)
    m = np.isin(y[a], [0, 1])
    mdl = lgb.LGBMClassifier(**{**T8_LGB, 'random_state': SEED, 'n_jobs': 1}).fit(Fa[m], y[a][m])
    return ('t8', 0, k, mdl.predict_proba(Fb)[:, 1], mdl.predict_proba(Ft)[:, 1])


def combine_ns(base, pns, w):
    """On rows predicted normal/shading, replace the N/S split by w * specialist + (1 - w) * base split."""
    out = base.copy()
    ns = base[:, 0] + base[:, 1]
    rows = np.isin(base.argmax(1), [0, 1])
    q = w * pns + (1 - w) * base[:, 1] / np.maximum(ns, 1e-9)
    out[rows, 1] = (q * ns)[rows]; out[rows, 0] = ((1 - q) * ns)[rows]
    return out


if USE_T8_SPECIALIST:
    RUNS_T8 = run_parallel([delayed(run_fold_t8)(k) for k in range(K)])
    S_OOF, S_TEST = RUNS_T8[('t8', 0)]
    print(f'T8 specialist normal-vs-shading AUC (OOF): {roc_auc_score(y[y < 2], S_OOF[y < 2]):.4f}')
    prev, CUR = CUR, f'{CUR}_T8w{int(round(T8_W * 100))}'
    add_stage(CUR, combine_ns(STAGES[prev]['oof'], S_OOF, T8_W), combine_ns(STAGES[prev]['test'], S_TEST, T8_W))

13 fold fits in 3s
T8 specialist normal-vs-shading AUC (OOF): 0.9883
STAGE etB3_T2a40_T8w30             macroF1=0.9556 | norma=0.956 shadi=0.932 short=0.984 degra=0.907 open_=1.000
   labels changed by the voltage guard: OOF 0, test 0


## 11. The shape of the day
The rows are shuffled, but a solar day is a smooth curve. Plotted in (G, T, V, I), the nearest neighbours of a reading were
recorded seconds before or after it, and faults come in blocks of time. So neighbours should agree, and the position of a reading on the curve
says something that its four numbers do not.

Both steps below use sensor values only, never labels or ids, and treat a hidden training day exactly as they treat the test set. Because
the test set mixes several days, we also score hidden days mixed with two others: arbitrary ones (the hardest case) and similar ones (closer
to the test).

### 11.1 Position on the curve
Let `R` be current over expected current, and `N` the number of modules delivering voltage. Along most days the same sequence appears:
full current, a ramp down into shade, a flat stretch that is normal, a second ramp, then flat shade until dusk. A single reading cannot
tell the two flat stretches apart. Its place in the sequence can.

The features are local averages and slopes of R and N, the morning or afternoon branch, and the values a little higher and lower on the
same branch. A LightGBM learns normal against shading from them.

In [11]:
from scipy.spatial import cKDTree
USE_TRAJ = flag('NB_TRAJ', True)
TRAJ_PAR = RAW + ['P', 'I_over_G', 'P_over_G', 'V_tc', 'I_ratio', 'V_ratio', 'P_ratio', 'dV_frac', 'R_like', 'lowG', 'notrack']
TRAJ_LGB = dict(n_estimators=300, learning_rate=0.04, num_leaves=63, min_child_samples=100, subsample=0.6, subsample_freq=1,
                colsample_bytree=0.3, reg_lambda=5.0, verbose=-1)
TRIP = [(0, 1, 2), (3, 4, 5), (6, 7, 8), (9, 10, 11), (10, 11, 12)]          # mixed-day evaluation pools
TRIP2 = [(0, 1, 3), (4, 5, 12), (6, 7, 9), (2, 8, 10), (10, 11, 12)]         # second pooling, training augmentation only
SIM3 = {0: (0, 1, 3), 1: (0, 1, 3), 3: (0, 1, 3), 4: (3, 4, 5), 5: (3, 4, 5), 6: (4, 5, 6), 2: (0, 1, 2), 7: (0, 1, 7), 8: (8, 11, 12),
        9: (9, 10, 11), 10: (10, 11, 12), 11: (10, 11, 12), 12: (10, 11, 12)}  # pools of 3 similar days (closest to the test set)
first_pool = lambda d, pools: next(t for t in pools if d in t)


def v_basis(I, T):
    l = np.log(np.maximum(I, .3) / 10); dT = T - 25
    return np.c_[np.ones_like(I), dT, l, l * dT, I / 10, l ** 2]


def fit_v(df, lab):
    V, I, G, T = df[RAW].values.T
    R = I / (G * (0.01643 + 7.2e-5 * (T - 25)))
    m = (lab == 0) & (G > 400) & (np.abs(R - 1) < .2); w = m
    for _ in range(6):
        c = np.linalg.lstsq(v_basis(I[w], T[w]), V[w], rcond=None)[0]
        r = V - v_basis(I, T) @ c; w = m & (np.abs(r) < 3 * np.median(np.abs(r[m])))
    return c


def otsu(x):
    x = np.sort(x); n = len(x)
    if n < 8:
        return x[n // 2]
    c = np.cumsum(x); k = np.arange(1, n); m0 = c[:-1] / k; m1 = (c[-1] - c[:-1]) / (n - k)
    i = (k * (n - k) * (m1 - m0) ** 2).argmax()
    return (x[i] + x[i + 1]) / 2


def traj_feats(df, c):
    """features of one pool (one day, or several mixed days) from its shuffled sensor rows"""
    V, I, G, T = df[RAW].values.T; n = len(df); f = {}
    R = np.clip(I / (G * (0.01643 + 7.2e-5 * (T - 25))), 0, 3)
    N = np.clip(24 * V / np.clip(v_basis(I, T) @ c, 150, 400), 10, 30)
    f['Rp'], f['Np'] = R, N
    f['lGest'] = np.log(np.maximum(I / (0.01643 + 7.2e-5 * (T - 25)), 1))
    Z = np.c_[G / 25., T / .5, np.log(I) * 4, V / 25.]; dd, ix = cKDTree(Z).query(Z, k=61)
    Rs, Ns = R[ix[:, :9]].mean(1), N[ix[:, :9]].mean(1)
    f['Rs'], f['Ns'], f['Vsd'] = Rs, Ns, N[ix[:, :9]].std(1)
    f['dens'] = np.log(dd[:, 8] + 1e-3); f['dens25'] = np.log(dd[:, 25] + 1e-3); j = ix[:, :26]
    g = G[j] - G[j].mean(1, keepdims=True); t = T[j] - T[j].mean(1, keepdims=True)
    for nm, A in [('R', R), ('N', N)]:
        a = A[j] - A[j].mean(1, keepdims=True)
        f['sl' + nm + 'G'] = (g * a).sum(1) / ((g * g).sum(1) + 50); f['sl' + nm + 'T'] = (t * a).sum(1) / ((t * t).sum(1) + .3)
    f['slTG'] = (g * t).sum(1) / ((g * g).sum(1) + 50); f['Gsd'] = g.std(1); f['Tsd'] = t.std(1)
    f['R60'], f['N60'] = R[ix].mean(1), N[ix].mean(1); f['Nd'] = Ns - f['N60']; f['Rd'] = Rs - f['R60']
    Z2 = np.c_[G / 10., T / .25]; i2 = cKDTree(Z2).query(Z2, k=16)[1]
    P = np.percentile(N[i2], [10, 50, 90], axis=1); Q = np.percentile(R[i2], [10, 50, 90], axis=1)
    f['Nm'], f['Nqr'], f['Rm'], f['Rqr'] = P[1], P[2] - P[0], Q[1], Q[2] - Q[0]
    o = np.argsort(G); Gs = G[o]; gr = np.arange(40, 1101, 10.); th = []      # morning / afternoon branch
    for x in gr:
        lo, hi = np.searchsorted(Gs, [x - 40, x + 40]); th.append(otsu(T[o[lo:hi]]) if hi - lo > 8 else np.nan)
    th = pd.Series(th).interpolate(limit_direction='both').rolling(9, center=True, min_periods=1).median()
    th = th.rolling(5, center=True, min_periods=1).mean().values
    f['pm'] = (T > np.interp(G, gr, th)).astype(float); f['pm25'] = f['pm'][j].mean(1)
    pm = f['pm'] > .5; sl = {}
    for s in (False, True):
        q = (pm == s) & (G > 300) & (G < 850); sl[s] = np.polyfit(G[q], T[q], 1)[0] if q.sum() > 50 else .02
    for s, (a, b) in {'p40': (20, 60), 'p100': (60, 140), 'm40': (-60, -20), 'm100': (-140, -60)}.items():
        r1 = np.full(n, np.nan); r2 = r1.copy()
        for i in range(n):                                                     # R and N a bit higher / lower on the same branch
            lo, hi = np.searchsorted(Gs, [G[i] + a, G[i] + b]); jj = o[lo:hi]; jj = jj[pm[jj] == pm[i]]
            if len(jj) < 2:
                continue
            e = T[jj] - T[i] - sl[bool(pm[i])] * (G[jj] - G[i]); w = np.exp(-.5 * (e / 2.) ** 2) + 1e-6; w /= w.sum()
            r1[i] = w @ Rs[jj]; r2[i] = w @ Ns[jj]
        f['R' + s] = r1 - Rs; f['N' + s] = r2 - Ns
    G99 = np.percentile(G, 99); f['dGmax'] = G99 - G; f['Gfr'] = G / G99
    hi = G > G99 * .93; f['Nrel'] = Ns - np.median(Ns[hi]); f['Rrel'] = Rs - np.median(Rs[hi])
    return pd.DataFrame(f, index=df.index).astype('f4')


def run_fold_traj(k):
    a, b = folds[k] if k < K else (np.arange(len(tr)), None)
    d = days[b[0]]; c = fit_v(tr.iloc[a], y[a])
    pool = lambda t: traj_feats(tr.iloc[np.where(np.isin(days, t))[0]], c)
    own = pd.concat([pool((e,)) for e in DAYS]).loc[tr.index]
    tr_days = [e for e in DAYS if e != d]

    def mixed(pools):                                    # training days only: the held-out day is dropped from every pool
        out = []
        cache = {}
        for e in tr_days:
            t = tuple(x for x in first_pool(e, pools) if x != d)
            if t not in cache:
                cache[t] = pool(t)
            out.append(cache[t].loc[tr.index[days == e]])
        return pd.concat(out).loc[tr.index[a]]

    P = fold_features(a, 'train', np.arange(len(tr)))[TRAJ_PAR]; P.index = tr.index
    Pt = fold_features(a, 'test')[TRAJ_PAR]; Xte = pd.concat([Pt.reset_index(drop=True), traj_feats(te, c).reset_index(drop=True)], axis=1)
    m = np.isin(y[a], [0, 1]); join = lambda F: pd.concat([P.loc[F.index], F], axis=1)
    sets = {'own': [own.iloc[a]], 'own+p': [own.iloc[a], mixed(TRIP), mixed(TRIP2)]}
    ev = [join(own.iloc[b]), join(pool(first_pool(d, TRIP[:-1] if d != 12 else TRIP)).loc[tr.index[b]]), join(pool(SIM3[d]).loc[tr.index[b]]), Xte]
    out = [0] * 4
    for fr in sets.values():
        Xa = pd.concat([join(F)[m] for F in fr]); ya = np.concatenate([y[a][m]] * len(fr))
        mdl = lgb.LGBMClassifier(**{**TRAJ_LGB, 'random_state': SEED, 'n_jobs': 1}).fit(Xa, ya)
        out = [o + mdl.predict_proba(X)[:, 1] / len(sets) for o, X in zip(out, ev)]
    return ('traj', 0, k, np.c_[out[0], out[1], out[2]], out[3])


if USE_TRAJ:
    RUNS_TRAJ = run_parallel([delayed(run_fold_traj)(k) for k in range(K)])
    TRAJ_OOF, TRAJ_TEST = RUNS_TRAJ[('traj', 0)]          # OOF columns: held-out day alone / in a mixed pool / in a similar pool
    print('trajectory specialist AUC (OOF, alone / mixed pool / similar pool):',
          [round(roc_auc_score(y[y < 2], TRAJ_OOF[y < 2, i]), 4) for i in range(3)], '| T8:', round(roc_auc_score(y[y < 2], S_OOF[y < 2]), 4))

13 fold fits in 20s
trajectory specialist AUC (OOF, alone / mixed pool / similar pool): [0.9922, 0.9889, 0.9902] | T8: 0.9883


### 11.2 Three opinions, then a vote
The normal-or-shading decision is the plain average of three opinions: the base model, the shading specialist and the position model.
Class probabilities are then averaged over each reading's nearest neighbours, twenty times: ten neighbours with equal weight in our second
pick, twenty with closer ones counting more in the first.

In [12]:
GS_SCALE, GS_K, GS_A, GS_IT = (10., .3, 2., .2), 10, .95, 20      # ~10-s step scales of G, T, V, I


def graph_smooth(p, df):
    """kNN-graph smoothing on the (G, T, V, I) trajectory of ONE pool (a held-out day, or the whole test set). Label-free."""
    Z = df[['irradiance', 'panel_temp', 'dc_voltage', 'dc_current']].values / np.array(GS_SCALE)
    j = cKDTree(Z).query(Z, k=GS_K + 1)[1][:, 1:]
    q = p.astype(float).copy()
    for _ in range(GS_IT):
        q = (1 - GS_A) * p + GS_A * q[j].mean(1)
    return q


def gauss_smooth(p, df, k=20):
    """same vote over 20 neighbours, closer ones counting more (Gaussian in distance, width = median distance to the 20th)"""
    Z = df[['irradiance', 'panel_temp', 'dc_voltage', 'dc_current']].values / np.array(GS_SCALE)
    d, j = cKDTree(Z).query(Z, k=k + 1); d, j = d[:, 1:], j[:, 1:]
    W = np.exp(-.5 * (d / np.median(d[:, -1])) ** 2) + 1e-6; W /= W.sum(1, keepdims=True)
    q = p.astype(float).copy()
    for _ in range(GS_IT):
        q = (1 - GS_A) * p + GS_A * (W[:, :, None] * q[j]).sum(1)
    return q


def smooth_oof(P, pool_of=lambda d: (d,), smoother=graph_smooth):
    """smooth every held-out day inside its pool (default: the day alone) and keep that day's rows"""
    q = P.copy()
    for d in DAYS:
        r = np.where(np.isin(days, pool_of(d)))[0]; s = smoother(P[r], tr.iloc[r]); q[r[days[r] == d]] = s[days[r] == d]
    return q


POOLS = {'alone': lambda d: (d,), 'mixed 3-day pool': lambda d: first_pool(d, TRIP[:-1] if d != 12 else TRIP), 'similar 3-day pool': lambda d: SIM3[d]}
G_ = lambda p: voltage_guard(p, tr.dc_voltage)
base, tree_stage = STAGES['etB3_T2a40'], 'etB3_T2a40_T8w30'
add_stage(tree_stage + '_smooth', smooth_oof(STAGES[tree_stage]['oof']), graph_smooth(STAGES[tree_stage]['test'], te))
for nm, pf in list(POOLS.items())[1:]:
    report(f'  same, {nm}', G_(smooth_oof(STAGES[tree_stage]['oof'], pf)))
if USE_TRAJ:
    split = lambda P: P[:, 1] / np.maximum(P[:, 0] + P[:, 1], 1e-9)
    mix = lambda b, s, t: combine_ns(b, (split(b) + s + t) / 3, 1.0)
    test3 = mix(base['test'], S_TEST, TRAJ_TEST)
    add_stage('three_opinions', mix(base['oof'], S_OOF, TRAJ_OOF[:, 0]), test3)
    add_stage('three_opinions_smooth', smooth_oof(STAGES['three_opinions']['oof']), graph_smooth(test3, te))
    for i, (nm, pf) in list(enumerate(POOLS.items()))[1:]:
        report(f'  same, {nm}', G_(smooth_oof(mix(base['oof'], S_OOF, TRAJ_OOF[:, i]), pf)))
    add_stage('three_opinions_gauss', smooth_oof(STAGES['three_opinions']['oof'], smoother=gauss_smooth), gauss_smooth(test3, te))
    for i, (nm, pf) in list(enumerate(POOLS.items()))[1:]:
        report(f'  same, {nm}', G_(smooth_oof(mix(base['oof'], S_OOF, TRAJ_OOF[:, i]), pf, gauss_smooth)))


STAGE etB3_T2a40_T8w30_smooth      macroF1=0.9678 | norma=0.960 shadi=0.937 short=0.988 degra=0.955 open_=1.000
   labels changed by the voltage guard: OOF 1, test 0


  same, mixed 3-day pool           macroF1=0.9680 | norma=0.960 shadi=0.936 short=0.990 degra=0.955 open_=1.000


  same, similar 3-day pool         macroF1=0.9682 | norma=0.960 shadi=0.936 short=0.990 degra=0.955 open_=1.000
STAGE three_opinions               macroF1=0.9590 | norma=0.963 shadi=0.942 short=0.984 degra=0.906 open_=1.000
   labels changed by the voltage guard: OOF 0, test 0


STAGE three_opinions_smooth        macroF1=0.9699 | norma=0.964 shadi=0.943 short=0.988 degra=0.955 open_=1.000
   labels changed by the voltage guard: OOF 1, test 0


  same, mixed 3-day pool           macroF1=0.9679 | norma=0.960 shadi=0.935 short=0.990 degra=0.955 open_=1.000


  same, similar 3-day pool         macroF1=0.9691 | norma=0.962 shadi=0.939 short=0.990 degra=0.955 open_=1.000


STAGE three_opinions_gauss         macroF1=0.9703 | norma=0.963 shadi=0.940 short=0.988 degra=0.960 open_=1.000
   labels changed by the voltage guard: OOF 1, test 0


  same, mixed 3-day pool           macroF1=0.9687 | norma=0.961 shadi=0.936 short=0.988 degra=0.959 open_=1.000


  same, similar 3-day pool         macroF1=0.9703 | norma=0.963 shadi=0.941 short=0.988 degra=0.960 open_=1.000


## 12. Results

In [13]:
rows = []
for n, s in STAGES.items():
    pc = class_f1(s['oof'])
    r = dict(stage=n, macro_f1=pc.mean(), **dict(zip(LABELS, pc)))
    for d in (8, 9, 11, 12):                             # the 4 days that contain electrical faults
        m = days == d; present = sorted(set(y[m]))
        r[f'day{d}'] = f1_score(y[m], s['oof'][m].argmax(1), average='macro', labels=present)
    rows.append(r)
print('Pooled LODO OOF macro F1, per-class F1, and macro F1 on the fault days:')
print(pd.DataFrame(rows).set_index('stage').round(4).to_string())

for n in FINAL_STAGES:
    if n not in STAGES:
        continue
    OOF = STAGES[n]['oof']; pred = OOF.argmax(1); pc = class_f1(OOF)
    print('\n' + '=' * 100 + f'\nFINAL STAGE {n} - pooled LODO OOF macro F1: {pc.mean():.4f}\n')
    print(pd.DataFrame({'f1': pc.round(4), 'support': np.bincount(y, minlength=5)}, index=LABELS))
    cm = pd.DataFrame(confusion_matrix(y, pred, labels=range(5)), index=['T_' + l for l in LABELS],
                      columns=['P_' + l for l in LABELS])
    print('\nConfusion matrix (rows = true, cols = predicted):'); print(cm)
    err = [(cm.iloc[i, j], LABELS[i], LABELS[j]) for i in range(5) for j in range(5) if i != j and cm.iloc[i, j]]
    print('most frequent confusions (true -> predicted):', '; '.join(f'{a} -> {b}: {c}' for c, a, b in sorted(err, reverse=True)[:6]))
    per_day = []
    for d in DAYS:
        m = days == d; present = sorted(set(y[m]))
        per_day.append(dict(day=d, n=int(m.sum()), classes=','.join(LABELS[c][:5] for c in present),
                            macro_f1=round(f1_score(y[m], pred[m], average='macro', labels=present), 4),
                            acc=round(float((pred[m] == y[m]).mean()), 4)))
    print('\nPer held-out day (macro F1 over the classes present that day):')
    print(pd.DataFrame(per_day).to_string(index=False))

Pooled LODO OOF macro F1, per-class F1, and macro F1 on the fault days:
                         macro_f1  normal  shading  short_circuit  degradation  open_circuit    day8    day9   day11   day12
stage                                                                                                                       
etB3                       0.9493  0.9514   0.9258         0.9816       0.8879           1.0  0.8732  0.9162  0.9384  0.9748
etB3_T2a40                 0.9536  0.9518   0.9257         0.9836       0.9070           1.0  0.8743  0.9275  0.9443  0.9731
etB3_T2a40_T8w30           0.9556  0.9558   0.9317         0.9836       0.9070           1.0  0.8857  0.9320  0.9427  0.9748
etB3_T2a40_T8w30_smooth    0.9678  0.9602   0.9366         0.9877       0.9547           1.0  0.8949  0.9616  0.9589  0.9771
three_opinions             0.9590  0.9627   0.9423         0.9836       0.9064           1.0  0.8982  0.9423  0.9468  0.9767
three_opinions_smooth      0.9699  0.9642   0.9427   

## 13. Submission files
Test probabilities are the mean of the thirteen fold models, passed through the same steps as in validation. If the uploaded files are
present, they are compared row by row.

In [14]:
def make_submission(P):
    s = pd.DataFrame({'id': te.id.values, 'label': np.array(LABELS)[P.argmax(1)]})
    s = ss[['id']].merge(s, on='id', how='left')
    assert len(s) == len(ss) == len(te) and s.label.notna().all() and set(s.label) <= set(LABELS)
    return s


SUBS = {n: make_submission(s['test']) for n, s in STAGES.items()}
missing = [n for n in FINAL_STAGES if n not in SUBS]
if missing:
    print('WARNING: final stages not computed (switched off):', missing)
finals = [n for n in FINAL_STAGES if n in SUBS] or [CUR]
for i, n in enumerate(finals):
    f = 'submission.csv' if i == 0 else f'submission_{i + 1}.csv'
    SUBS[n].to_csv(f, index=False); print(f'{f} <- stage {n}')
np.save('final_oof_probs.npy', STAGES[finals[0]]['oof']); np.save('final_test_probs.npy', STAGES[finals[0]]['test'])

REF = {'three_opinions_gauss': 'p2_three_opinions_gauss.csv',        # final pick 1, public LB 0.954
       'three_opinions_smooth': 'p2_three_opinions_smooth.csv'}      # final pick 2, public LB 0.954
for n, f in REF.items():
    p = Path('../submissions') / f
    if n in SUBS and not ON_KAGGLE and p.exists():
        j = pd.read_csv(p).merge(SUBS[n], on='id', suffixes=('_ref', ''))
        print(f'reproducibility: stage {n}: {(j.label_ref != j.label).sum()} label mismatches vs {f} ({len(j)} rows compared)')
print('\ntest label counts per stage:')
print(pd.DataFrame({n: s.label.value_counts().reindex(LABELS) for n, s in SUBS.items()}).T)
print(f'\nTotal wall time: {time.time() - T0:.0f}s (n_jobs={N_JOBS}, cores={os.cpu_count()})')

submission.csv <- stage three_opinions_gauss
submission_2.csv <- stage three_opinions_smooth
reproducibility: stage three_opinions_gauss: 0 label mismatches vs p2_three_opinions_gauss.csv (9617 rows compared)
reproducibility: stage three_opinions_smooth: 0 label mismatches vs p2_three_opinions_smooth.csv (9617 rows compared)

test label counts per stage:
label                    normal  shading  short_circuit  degradation  open_circuit
etB3                       5537     3036            362          322           360
etB3_T2a40                 5517     3036            359          345           360
etB3_T2a40_T8w30           5546     3006            360          345           360
etB3_T2a40_T8w30_smooth    5517     3024            363          353           360
three_opinions             5610     2942            360          345           360
three_opinions_smooth      5591     2950            363          353           360
three_opinions_gauss       5584     2957            363       

## 14. Summary
| Pipeline | LODO macro F1 | Kaggle public |
|---|---|---|
| Raw sensors, LightGBM | 0.780 | |
| Physics features | 0.8753 | |
| Two-stage blend | 0.9493 | |
| Synthetic faults | 0.9536 | 0.942 |
| Shading specialist | 0.9556 | |
| Neighbour voting | 0.9678 | 0.950 |
| Three opinions (final pick 2) | 0.9699 | 0.954 |
| **Distance-weighted voting (final pick 1)** | **0.9703** | **0.954** |

With days mixed as in the test set, the last three score 0.9680 to 0.9682, 0.9679 to 0.9691 and 0.9687 to 0.9703. The final steps are
modest. Nearly all remaining errors are normal against shading.

**AI disclosure.** Claude Code (Anthropic) wrote most of the code, the experiments and the first draft of the text, and did the literature search. The team's
own decisions and the chat links are in the README.